In [2]:
# to cut the pages

import pandas as pd
from PyPDF2 import PdfReader, PdfWriter
from tqdm import tqdm

# === CONFIGURATION ===
excel_file = r"pages.xlsx"      # Excel file with page numbers
pdf_file = r"C:\Users\USER\Downloads\ilovepdf_merged (1).pdf"         # Your large 3000-page PDF
output_pdf = r"CO 01.pdf"  # Output PDF with Excel order

# Read numbers from Excel
df = pd.read_excel(excel_file)
search_terms = df.iloc[:, 0].dropna().astype(str).tolist()

# Load PDF
reader = PdfReader(pdf_file)
writer = PdfWriter()

print(f"🔍 Searching for {len(search_terms)} terms across {len(reader.pages)} pages...")

# Map each search term to its matching page number
term_page_map = {}

for i, page in enumerate(tqdm(reader.pages, desc="Scanning PDF pages")):
    text = page.extract_text() or ""
    for term in search_terms:
        if term not in term_page_map and term in text:
            term_page_map[term] = i  # store first page found

# Add pages in Excel order
for term in search_terms:
    if term in term_page_map:
        page_index = term_page_map[term]
        writer.add_page(reader.pages[page_index])

# Write output PDF
with open(output_pdf, "wb") as f:
    writer.write(f)

print(f"✅ Done! Extracted {len(writer.pages)} pages to '{output_pdf}' in Excel order.")


🔍 Searching for 43 terms across 1792 pages...


Scanning PDF pages: 100%|██████████████████████████████████████████████████████████| 1792/1792 [09:49<00:00,  3.04it/s]

✅ Done! Extracted 7 pages to 'CO 01.pdf' in Excel order.


In [17]:
# verify pages details
import pandas as pd
from PyPDF2 import PdfReader
from tqdm import tqdm
import re

excel_file = r"GV.xlsx"
pdf_file = r"CO 30.pdf"
output_excel = r"GV OUTPUT.xlsx"

df = pd.read_excel(excel_file)

reader = PdfReader(pdf_file)

def exact_word_match(word, text):
    pattern = r"\b" + re.escape(word.lower()) + r"\b"
    return re.search(pattern, text) is not None

results = []

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing Rows"):
    colA = str(row.iloc[0]).strip()
    colB = str(row.iloc[1]).strip()

    foundA_page = []
    foundB_page = []

    for page_no, page in enumerate(reader.pages, start=1):
        text = page.extract_text()
        if not text:
            continue

        text_lower = text.lower()

        if exact_word_match(colA, text_lower):
            foundA_page.append(page_no)

        if exact_word_match(colB, text_lower):
            foundB_page.append(page_no)

    if foundA_page and foundB_page:
        result = f"✅ FOUND A & B (Pages: {sorted(set(foundA_page + foundB_page))})"
    elif foundA_page:
        result = f"✅ FOUND A (Pages: {foundA_page})"
    elif foundB_page:
        result = f"✅ FOUND B (Pages: {foundB_page})"
    else:
        result = "❌ NOT FOUND"

    results.append(result)

df["Verification"] = results
df.to_excel(output_excel, index=False, engine="openpyxl")

print(f"✅ Done! Results saved in '{output_excel}'")

Processing Rows: 100%|█████████████████████████████████████████████████████████████████| 43/43 [20:04<00:00, 28.01s/it]

✅ Done! Results saved in 'GV OUTPUT.xlsx'


In [13]:
import fitz  # PyMuPDF

input_pdf = r"CE 30.pdf"
output_pdf = r"TEMP.pdf"

doc = fitz.open(input_pdf)
new_doc = fitz.open()

# Define crop boxes (x0, y0, x1, y1)
# IMPORTANT: adjust if your coordinates represent different corners
crop_areas = [
    (496,32,572,120),   # from your points (assumed rectangle)
    (20,380,570,519)   # example second area (edit as needed)
]

for page in doc:
    for rect in crop_areas:
        x0, y0, x1, y1 = rect

        clip = fitz.Rect(x0, y0, x1, y1)

        # Create new page with same size as crop
        new_page = new_doc.new_page(width=clip.width, height=clip.height)

        # Copy cropped region into new page
        new_page.show_pdf_page(
            fitz.Rect(0, 0, clip.width, clip.height),
            doc,
            page.number,
            clip=clip
        )

new_doc.save(output_pdf)
new_doc.close()
doc.close()

print("Done! Cropped PDF saved.")

Done! Cropped PDF saved.
